# 02 · Corbes de Kaplan-Meier i test log-rank

L'estimador de Kaplan-Meier estima la funció de supervivència $S(t) = P(T > t)$ sense supòsits
sobre la seva forma i tenint en compte la censura per la dreta. El test log-rank contrasta si
les corbes de diversos grups són iguals.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.precision", 3)
from src.preparacio import carregar_dades
from src.km import (ajustar_km, afegir_quartils_quota, logrank, taula_medianes, taula_resum_km,
                    VARIABLES_KM, NOMS_VARIABLES)
from src.grafics import fig_km, exportar
df = afegir_quartils_quota(carregar_dades())

## Corba global

In [2]:
km_global = ajustar_km(df)
fig = fig_km(km_global, "Supervivència global dels clients (Kaplan-Meier)")
exportar(fig, "km_global"); fig.show()
taula_medianes(km_global)

,grup,n,mediana,S_6m,S_12m,S_24m
0,Global,7032,no assolida,0.885,0.843,0.789


La mediana global **no s'assoleix**: més de la meitat dels clients continuen actius als
72 mesos. Per això és més informatiu llegir $S(t)$ en horitzons concrets (6, 12, 24 mesos).

## Comparació per segments: quines variables separen més les corbes?

In [3]:
resum = taula_resum_km(df)
resum.assign(variable=resum.variable.map(NOMS_VARIABLES))

,variable,estadistic,p_valor
0,Tipus de contracte,2352.873,0.000e+00
1,Seguretat en línia,1013.865,6.951e-221
2,Suport tècnic,989.560,1.317e-215
3,Mètode de pagament,865.239,3.067e-187
4,Servei d'internet,520.121,1.141e-113
5,Té parella,423.543,4.133e-94
6,Quota mensual (quartils),236.834,4.603e-51
7,Client sènior,109.490,1.268e-25


## Tipus de contracte (la variable més discriminant)

In [4]:
km_contracte = ajustar_km(df, "Contract")
fig = fig_km(km_contracte, "Supervivència per tipus de contracte")
exportar(fig, "km_contracte"); fig.show()
print(logrank(df, "Contract"))
taula_medianes(km_contracte)

{'estadistic': 2352.8725382593407, 'p_valor': 0.0}


,grup,n,mediana,S_6m,S_12m,S_24m
0,Month-to-month,3875,35,0.786,0.703,0.586
1,One year,1472,no assolida,0.997,0.991,0.978
2,Two year,1685,no assolida,1.000,1.000,1.000


## Resta de variables

In [5]:
for var in VARIABLES_KM:
    if var == "Contract":
        continue
    fitters = ajustar_km(df, var)
    fig = fig_km(fitters, f"Supervivència per: {NOMS_VARIABLES[var].lower()}", interval=False)
    exportar(fig, f"km_{var}")
    fig.show()

## Conclusions

- **Contracte**: el factor més potent (log-rank $\chi^2 \approx 2.350$, $p \approx 0$). Els clients
  mensuals tenen una supervivència mediana de **35 mesos** i només un **70 %** continua al cap
  d'un any; amb contracte d'1 o 2 anys la mediana no s'assoleix i $S(12) \geq 99\%$.
- **Seguretat en línia, suport tècnic, mètode de pagament i servei d'internet** (per ordre) també separen
  clarament les corbes: la fibra òptica i el xec electrònic s'associen a més abandonament; els
  serveis de suport, a menys.
- Kaplan-Meier és **univariant**: no separa efectes que van junts (p. ex. fibra ↔ quota alta).
  Per això el següent pas és el model de Cox.